In [1]:
import numpy as np

eps = 1e-10
kmax = 1000

In [2]:
def find_max_offdiag(A):
    n = A.shape[0]
    mask = np.tril(np.ones((n, n), dtype=bool), -1)
    idx = np.argmax(np.abs(A[mask]))
    rows, cols = np.where(mask)
    return rows[idx], cols[idx]


def compute_cs(A, p, q):
    # alpha = (a_pp - a_qq) / (2 * a_pq)
    alpha = (A[p, p] - A[q, q]) / (2.0 * A[p, q])

    # t = -alpha + semn(alpha) * sqrt(alpha^2 + 1)
    t = -alpha + np.sqrt(alpha**2 + 1) if alpha >= 0 else -alpha - np.sqrt(alpha**2 + 1)

    # c = 1 / sqrt(1 + t^2),  s = t / sqrt(1 + t^2)
    c = 1.0 / np.sqrt(1.0 + t**2)
    s = t * c
    return c, s, t


def jacobi(A, eps=eps, kmax=kmax):
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError(f"Jacobi: A trebuie sa fie patrata, primit {A.shape}.")
    if not np.allclose(A, A.T):
        raise ValueError("Jacobi: A trebuie sa fie simetrica (A = A^T).")

    n = A.shape[0]
    A = A.copy().astype(float)
    # U = I_n
    U = np.eye(n)
    k = 0
    # p, q = argmax |a_ij|, i != j, cautare in triunghiul inferior
    p, q = find_max_offdiag(A)

    # while |a_pq| > eps si k <= kmax
    while abs(A[p, q]) > eps and k <= kmax:
        c, s, t = compute_cs(A, p, q)
        apq = A[p, q]

        mask = np.ones(n, dtype=bool)
        mask[p] = mask[q] = False
        apj = A[p, mask].copy()
        aqj = A[q, mask].copy()
        # a_pj = c*a_pj + s*a_qj,  j != p, j != q
        A[p, mask] = c * apj + s * aqj
        A[mask, p] = A[p, mask]
        # a_qj = -s*a_pj + c*a_qj,  j != p, j != q
        A[q, mask] = -s * apj + c * aqj
        A[mask, q] = A[q, mask]

        # a_pp = a_pp + t*a_pq
        A[p, p] = A[p, p] + t * apq
        # a_qq = a_qq - t*a_pq
        A[q, q] = A[q, q] - t * apq
        # a_pq = a_qp = 0
        A[p, q] = A[q, p] = 0.0

        uip = U[:, p].copy()
        uiq = U[:, q].copy()
        # u_ip = c*u_ip + s*u_iq
        U[:, p] = c * uip + s * uiq
        # u_iq = -s*u_ip_vechi + c*u_iq
        U[:, q] = -s * uip + c * uiq

        # recalculeaza p, q
        p, q = find_max_offdiag(A)
        k += 1

    eigenvalues = np.diag(A).copy()
    return eigenvalues, U

In [3]:
matrices = {
    "Ex 1 (lambda = -1, 0, 2)": np.array(
        [[0, 0, 1], [0, 0, 1], [1, 1, 1]], dtype=float
    ),
    "Ex 2 (lambda = -0.828, 0, 4.828)": np.array(
        [[1, 1, 2], [1, 1, 2], [2, 2, 2]], dtype=float
    ),
    "Ex 3 (lambda = 0, 0, 2, 2)": np.array(
        [[1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1]], dtype=float
    ),
    "Ex 4 (lambda = -1.165, 0, 0, 17.165)": np.array(
        [[1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 5, 6], [4, 5, 6, 7]], dtype=float
    ),
}

for label, A in matrices.items():
    A_init = A.copy()
    eigenvalues, U = jacobi(A)

    idx = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[idx]
    U = U[:, idx]
    Lambda = np.diag(eigenvalues)

    print(f"=== {label} ===")
    print(f"  eigenvalues = {np.round(eigenvalues, 8)}")

    print(
        f"  ||A_init * U - U * Lambda|| = {np.linalg.norm(A_init @ U - U @ Lambda):.2e}"
    )
    print()

=== Ex 1 (lambda = -1, 0, 2) ===
  eigenvalues = [-1. -0.  2.]
  ||A_init * U - U * Lambda|| = 6.35e-10

=== Ex 2 (lambda = -0.828, 0, 4.828) ===
  eigenvalues = [-0.82842712  0.          4.82842712]
  ||A_init * U - U * Lambda|| = 1.54e-09

=== Ex 3 (lambda = 0, 0, 2, 2) ===
  eigenvalues = [0. 0. 2. 2.]
  ||A_init * U - U * Lambda|| = 0.00e+00

=== Ex 4 (lambda = -1.165, 0, 0, 17.165) ===
  eigenvalues = [-1.16515139 -0.         -0.         17.16515139]
  ||A_init * U - U * Lambda|| = 1.46e-08



In [ ]:
def cholesky_iteration(A, eps=eps, kmax=kmax):
    if not np.allclose(A, A.T):
        raise ValueError("Cholesky: A trebuie sa fie simetrica (A = A^T).")

    # A^(0) = A
    A_prev = A.copy().astype(float)
    A_curr = None
    k = 0

    while k < kmax:
        try:
            # A^(k) = Lk * Lk^T  (factorizare Cholesky)
            L = np.linalg.cholesky(A_prev)
        except np.linalg.LinAlgError:
            raise np.linalg.LinAlgError(
                f"Cholesky: A^({k}) nu este pozitiv definita; "
                "algoritmul necesita A simetrica si pozitiv definita."
            )

        # A^(k+1) = Lk^T * Lk
        A_curr = L.T @ L

        # criteriu de oprire: ||A^(k) - A^(k-1)|| < eps
        if np.linalg.norm(A_curr - A_prev) < eps:
            k += 1
            break

        A_prev = A_curr
        k += 1

    return A_curr if A_curr is not None else A_prev, k

In [5]:
np.random.seed(42)
n = 4
M = np.random.rand(n, n)
Ac = M.T @ M + n * np.eye(n)
Ac2 = np.array([[4, 2, 0], [2, 3, 1], [0, 1, 2]], dtype=float)

for label, A in [("4x4 random SPD", Ac), ("3x3 pozitiv definita", Ac2)]:
    A_final, k = cholesky_iteration(A)
    print(f"=== {label} ({k} iteratii) ===")
    print(f"  A_final =\n{np.round(A_final, 8)}")
    print(f"  Forma: matrice diagonala (off-diagonal -> 0)")
    print(f"  Informatii: diag(A_final) = valorile proprii ale lui A")
    print(f"  diag(A_final) = {np.round(np.sort(np.diag(A_final))[::-1], 8)}")
    print()

=== 4x4 random SPD (1000 iteratii) ===
  A_final =
[[ 8.28701606e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00]
 [-0.00000000e+00  4.54967272e+00  1.25800000e-05  0.00000000e+00]
 [-0.00000000e+00  1.25800000e-05  4.44528924e+00 -0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00 -0.00000000e+00  4.06727908e+00]]
  Forma: matrice diagonala (off-diagonal -> 0)
  Informatii: diag(A_final) = valorile proprii ale lui A
  diag(A_final) = [8.28701606 4.54967272 4.44528924 4.06727908]

=== 3x3 pozitiv definita (57 iteratii) ===
  A_final =
[[5.66907909 0.         0.        ]
 [0.         2.4760236  0.        ]
 [0.         0.         0.85489731]]
  Forma: matrice diagonala (off-diagonal -> 0)
  Informatii: diag(A_final) = valorile proprii ale lui A
  diag(A_final) = [5.66907909 2.4760236  0.85489731]



In [6]:
def svd_tasks(A):
    if A.ndim != 2:
        raise ValueError(f"SVD: A trebuie sa fie o matrice 2D, primit {A.ndim}D.")
    p, n = A.shape
    if p <= n:
        raise ValueError(f"SVD: necesita p > n, primit ({p}, {n}).")

    # A = U * S * V^T
    U, S, VT = np.linalg.svd(A, full_matrices=True)
    V = VT.T

    singular_values = S

    # rang(A) = numarul de valori singulare strict pozitive
    rank = int(np.sum(S > eps))
    rank_lib = np.linalg.matrix_rank(A)

    # k2(A) = sigma_max / sigma_min_pozitiva
    cond_number = S[0] / S[S > eps][-1]
    cond_number_lib = np.linalg.cond(A)

    # S^I: 1/sigma_i pe diagonala pentru i <= rang, 0 altfel
    SI = np.zeros((n, p))
    SI[:rank, :rank] = np.diag(1.0 / S[:rank])
    # A^I = V * S^I * U^T
    A_I = V @ SI @ U.T

    # A^J = (A^T * A)^-1 * A^T
    ATA = A.T @ A
    try:
        A_J = np.linalg.inv(ATA) @ A.T
        # ||A^I - A^J||_1
        norm_diff = np.linalg.norm(A_I - A_J, ord=1)
    except np.linalg.LinAlgError:
        raise np.linalg.LinAlgError(
            f"SVD: A^T*A este singulara (rang(A) = {rank} < {n} = n); "
            "formula A_J = (A^T A)^-1 A^T necesita rang plin pe coloane."
        )

    return (
        singular_values,
        rank,
        rank_lib,
        cond_number,
        cond_number_lib,
        A_I,
        A_J,
        norm_diff,
    )

In [7]:
B1 = np.array([[1, 2], [3, 4], [5, 6], [7, 8]], dtype=float)

np.random.seed(7)
B2 = np.round(np.random.rand(5, 3) * 10).astype(float)

B3 = np.array(
    [
        [1, 2, 3, 1],
        [2, 4, 1, 2],
        [3, 6, 2, 3],
        [4, 8, 0, 4],
        [5, 10, 1, 5],
        [1, 2, 4, 1],
    ],
    dtype=float,
)

for label, A in [
    ("p=4, n=2, rang plin", B1),
    ("p=5, n=3, aleatoare", B2),
    ("p=6, n=4, rang deficient", B3),
]:
    print(f"=== {label} ===")
    try:
        (
            singular_values,
            rank,
            rank_lib,
            cond_number,
            cond_number_lib,
            A_I,
            A_J,
            norm_diff,
        ) = svd_tasks(A)

        print(f"  singular_values = {np.round(singular_values, 6)}")
        print(f"  rank            = {rank}  (numpy: {rank_lib})")
        print(f"  cond_number     = {cond_number:.6f}  (numpy: {cond_number_lib:.6f})")
        print(f"  A_I =\n{np.round(A_I, 6)}")
        print(f"  A_J =\n{np.round(A_J, 6)}")
        print(f"  ||A_I - A_J||_1 = {norm_diff:.2e}")
    except (ValueError, np.linalg.LinAlgError) as e:
        print(f"  Eroare: {e}")
    print()

=== p=4, n=2, rang plin ===
  singular_values = [14.269095  0.626828]
  rank            = 2  (numpy: 2)
  cond_number     = 22.763964  (numpy: 22.763964)
  A_I =
[[-1.   -0.5   0.    0.5 ]
 [ 0.85  0.45  0.05 -0.35]]
  A_J =
[[-1.   -0.5   0.    0.5 ]
 [ 0.85  0.45  0.05 -0.35]]
  ||A_I - A_J||_1 = 1.12e-14

=== p=5, n=3, aleatoare ===
  singular_values = [20.22041   5.955497  3.109191]
  rank            = 3  (numpy: 3)
  cond_number     = 6.503431  (numpy: 6.503431)
  A_I =
[[-0.093553  0.117221  0.09371  -0.081498  0.052986]
 [ 0.067238  0.09463  -0.04989  -0.054655 -0.051731]
 [ 0.034425 -0.171156  0.00204   0.199261  0.03929 ]]
  A_J =
[[-0.093553  0.117221  0.09371  -0.081498  0.052986]
 [ 0.067238  0.09463  -0.04989  -0.054655 -0.051731]
 [ 0.034425 -0.171156  0.00204   0.199261  0.03929 ]]
  ||A_I - A_J||_1 = 2.78e-16

=== p=6, n=4, rang deficient ===
  Eroare: SVD: A^T*A este singulara (rang(A) = 2 < 4 = n); formula A_J = (A^T A)^-1 A^T necesita rang plin pe coloane.

